# Lab 8 – Task B: Drift

**5-phase state machine on Arduino:**
- Phase 0 APPROACH: full-speed forward + KF distance tracking
- Phase 1 BRAKE: hard reverse at `BRAKE_PWM` for `BRAKE_MS` ms (rapid decel)
- Phase 2 ROTATE: orient-PID turns exactly 180° using IMU reference locked at brake entry
- Phase 3 RETURN: full-speed forward (robot now faces away from wall)
- Phase 4 DONE: stopped

## 1 – Imports & BLE setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import importlib

from ble import get_ble_controller
from base_ble import LOG
import cmd_types
importlib.reload(cmd_types)
from cmd_types import CMD

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['font.size'] = 12

print('Imports done.')

In [ ]:
ble = get_ble_controller()
ble.connect()
print('Connected!')

def ble_reconnect():
    global ble
    print('Reconnecting BLE ...')
    try:
        ble.disconnect()
    except Exception:
        pass
    time.sleep(1.0)
    ble = get_ble_controller()
    ble.connect()
    print('Reconnected!')

## 2 – Parameters

Tune these before each run.

In [ ]:
# ── Drift motion ──────────────────────────────────────────────────────────
APPROACH_PWM = 200       # PWM during approach (0-255)
RETURN_PWM   = 200       # PWM during return  (0-255)
BRAKE_DIST   = 914       # mm — 3 ft, KF estimate that triggers braking (phase 0→1)
BRAKE_PWM    = 200       # reverse PWM for hard deceleration
BRAKE_MS     = 250       # ms of reverse braking
RETURN_MS    = 2500      # ms — how long to drive back after rotation
TIMEOUT_MS   = 10000     # ms — global safety timeout

# ── Kalman Filter (from Lab 7 system ID) ─────────────────────────────────
KF_D   = 0.0003163
KF_M   = 0.0002093
KF_S1  = 50.0
KF_S2  = 50.0
KF_S3  = 20.0

# ── Orientation PID (from Lab 6) ──────────────────────────────────────────
ORIENT_KP = 2.5
ORIENT_KI = 0.0
ORIENT_KD = 0.05

print('Parameters set.')

## 3 – Configure robot

In [ ]:
# KF parameters (step_pwm = APPROACH_PWM for normalization base)
ble.send_command(CMD.SET_KF_PARAMS,
    f"{KF_D}|{KF_M}|{KF_S1}|{KF_S2}|{KF_S3}|{APPROACH_PWM}")
time.sleep(0.15)
print('KF:', ble.receive_string(ble.uuid['RX_STRING']))

# Orientation PID gains
ble.send_command(CMD.SET_ORIENT_GAINS,
    f"{ORIENT_KP}|{ORIENT_KI}|{ORIENT_KD}")
time.sleep(0.15)
print('Orient gains:', ble.receive_string(ble.uuid['RX_STRING']))

# Pre-set all drift parameters without starting
# Format: approach_pwm|return_pwm|brake_dist|brake_pwm|brake_ms|return_ms|timeout_ms
ble.send_command(CMD.SET_DRIFT_PARAMS,
    f"{APPROACH_PWM}|{RETURN_PWM}|{BRAKE_DIST}|{BRAKE_PWM}|{BRAKE_MS}|{RETURN_MS}|{TIMEOUT_MS}")
time.sleep(0.15)
print('Drift params:', ble.receive_string(ble.uuid['RX_STRING']))

## 4 – Run drift

Position robot at starting line (≥ 4 m from wall), then execute the cell below.

In [ ]:
raw_messages = []
_drift_done  = False

def drift_notify_handler(uuid, bytearray_data):
    global _drift_done
    try:
        msg = ble.bytearray_to_string(bytearray_data).strip()
        raw_messages.append(msg)
        if msg.startswith('DRF_END'):
            _drift_done = True
    except Exception as ex:
        print(f'Handler error: {ex}')

# ── Start run ────────────────────────────────────────────────────────────
raw_messages.clear()
_drift_done = False

try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception:
    pass
ble.start_notify(ble.uuid['RX_STRING'], drift_notify_handler)
time.sleep(0.05)

input('Place robot ≥4 m from wall. Press Enter to START ...')

# Format: approach_pwm|return_pwm|brake_dist|brake_pwm|brake_ms|return_ms|timeout_ms
ble.send_command(CMD.DRIFT_START,
    f"{APPROACH_PWM}|{RETURN_PWM}|{BRAKE_DIST}|{BRAKE_PWM}|{BRAKE_MS}|{RETURN_MS}|{TIMEOUT_MS}")

print(f'Drift running ... waiting {TIMEOUT_MS/1000 + 1:.0f}s')
time.sleep(TIMEOUT_MS / 1000 + 1.5)

# Retrieve logged data
print('Fetching data ...')
_drift_done = False
ble.send_command(CMD.GET_DRIFT_DATA, '')

t0 = time.time()
while not _drift_done and (time.time() - t0 < 30.0):
    time.sleep(0.1)

ble.stop_notify(ble.uuid['RX_STRING'])
print(f'Received {len(raw_messages)} messages  (DRF_END: {_drift_done})')

## 5 – Parse data

In [ ]:
def parse_drift_row(s):
    """Parse 'DRF|raw|est|yaw_x10|mot|phase|ts' → dict."""
    parts = s.split('|')
    return {
        'raw_mm':  int(parts[1]),
        'est_mm':  int(parts[2]),
        'yaw_deg': int(parts[3]) / 10.0,
        'motor':   int(parts[4]),
        'phase':   int(parts[5]),
        'ts_ms':   int(parts[6]),
    }

drf_rows  = [parse_drift_row(m) for m in raw_messages if m.startswith('DRF|')]
ctrl_msgs = [m for m in raw_messages if not m.startswith('DRF|')]

df = pd.DataFrame(drf_rows)
if not df.empty:
    df['ts_ms']    -= df['ts_ms'].iloc[0]       # zero-base timestamps
    df['raw_valid'] = df['raw_mm'] > 0

print(f'Parsed {len(df)} data rows.')
print('Control messages:', ctrl_msgs)
df.head()

## 6 – Plot

In [ ]:
if df.empty:
    print('No data to plot.')
else:
    phase_colors = {0: 'steelblue', 1: 'crimson', 2: 'darkorange', 3: 'seagreen'}
    phase_labels = {0: 'Approach', 1: 'Brake', 2: 'Rotate', 3: 'Return'}

    # First timestamp per phase
    phase_starts = {}
    for ph in [0, 1, 2, 3]:
        rows = df[df['phase'] == ph]
        if not rows.empty:
            phase_starts[ph] = rows['ts_ms'].iloc[0]

    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    # ── Distance ─────────────────────────────────────────────────────────
    ax = axes[0]
    raw = df[df['raw_valid']]
    ax.plot(raw['ts_ms'], raw['raw_mm'],
            'b.', alpha=0.4, ms=4, label='Raw TOF')
    est = df[df['est_mm'] > 0]
    ax.plot(est['ts_ms'], est['est_mm'],
            'r-', lw=1.5, label='KF Estimate')
    ax.axhline(BRAKE_DIST, color='crimson', ls='--', lw=1.2,
               label=f'Brake trigger ({BRAKE_DIST} mm)')
    for ph, t in phase_starts.items():
        ax.axvline(t, color=phase_colors[ph], ls=':', lw=1.2,
                   label=f'Ph{ph}: {phase_labels[ph]}')
    ax.set_ylabel('Distance (mm)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title('Lab 8 Drift – Distance')

    # ── Yaw ──────────────────────────────────────────────────────────────
    ax = axes[1]
    ax.plot(df['ts_ms'], df['yaw_deg'], 'g-', lw=1.5, label='Yaw (deg)')
    # Compute target yaw from data (yaw at end of brake phase / start of rotate phase)
    if 2 in phase_starts:
        brake_end_rows = df[df['phase'] == 1]
        if not brake_end_rows.empty:
            yaw_at_brake  = brake_end_rows['yaw_deg'].iloc[0]
            yaw_target    = yaw_at_brake + 180 if yaw_at_brake + 180 <= 180 else yaw_at_brake - 180
            ax.axhline(yaw_target, color='orange', ls='--', lw=1.2,
                       label=f'Target ({yaw_target:.1f}°)')
    for ph, t in phase_starts.items():
        ax.axvline(t, color=phase_colors[ph], ls=':', lw=1.2)
    ax.set_ylabel('Yaw (deg)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title('Yaw Angle')

    # ── Motor PWM ────────────────────────────────────────────────────────
    ax = axes[2]
    ax.plot(df['ts_ms'], df['motor'], 'k-', lw=1.2, label='Motor PWM')
    ax.axhline(0, color='gray', ls='-', lw=0.5)
    for ph, t in phase_starts.items():
        ax.axvline(t, color=phase_colors[ph], ls=':', lw=1.2,
                   label=f'Ph{ph}: {phase_labels[ph]}')
    ax.set_ylabel('PWM')
    ax.set_xlabel('Time (ms)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_title('Motor Command')

    plt.tight_layout()
    plt.savefig('lab8_drift.png', dpi=150)
    plt.show()
    print('Saved lab8_drift.png')

## 7 – Diagnostics

In [ ]:
if not df.empty:
    total_ms  = df['ts_ms'].iloc[-1] - df['ts_ms'].iloc[0]
    ph_counts = df['phase'].value_counts().sort_index()
    print(f'Total run time  : {total_ms} ms')
    print(f'Rows per phase  : {ph_counts.to_dict()}')

    # Approach
    if 0 in ph_counts.index:
        approach = df[df['phase'] == 0]
        kf_at_brake = approach['est_mm'].iloc[-1]
        print(f'\n[Phase 0 → 1] KF dist at brake trigger : {kf_at_brake:.0f} mm  (target {BRAKE_DIST} mm)')

    # Brake
    if 1 in ph_counts.index:
        brake = df[df['phase'] == 1]
        brake_duration = brake['ts_ms'].iloc[-1] - brake['ts_ms'].iloc[0]
        yaw_at_brake   = brake['yaw_deg'].iloc[0]
        print(f'[Phase 1] Brake duration    : {brake_duration} ms  (target {BRAKE_MS} ms)')
        print(f'[Phase 1] Yaw at brake entry: {yaw_at_brake:.1f}°')
        print(f'[Phase 1] IMU rotation target: {yaw_at_brake + 180:.1f}° (or {yaw_at_brake - 180:.1f}°)')

    # Rotate
    if 2 in ph_counts.index:
        rotate     = df[df['phase'] == 2]
        yaw_start  = rotate['yaw_deg'].iloc[0]
        yaw_end    = rotate['yaw_deg'].iloc[-1]
        delta      = yaw_end - yaw_start
        rotate_ms  = rotate['ts_ms'].iloc[-1] - rotate['ts_ms'].iloc[0]
        print(f'[Phase 2] Yaw: {yaw_start:.1f}° → {yaw_end:.1f}°  (delta {delta:.1f}°, target ±180°)')
        print(f'[Phase 2] Rotation duration : {rotate_ms} ms')

## 8 – Emergency stop

In [ ]:
ble.send_command(CMD.DRIFT_STOP, '')
time.sleep(0.1)
print('Emergency stop sent.')

## 9 – Tuning notes

| Issue | Suggested fix |
|-------|---------------|
| Brakes too late / hits wall | Decrease `BRAKE_DIST` (try 600-700 mm) |
| Brakes too early / doesn't drift | Increase `BRAKE_DIST` (try 400 mm) |
| Still moving when rotation starts | Increase `BRAKE_MS` (try 350 ms) |
| Overshoots 180° | Increase `ORIENT_KD` |
| Undershoots / slow rotation | Increase `ORIENT_KP` |
| Doesn't return past start line | Increase `RETURN_MS` or `RETURN_PWM` |
| KF fires brake too early | Re-run Lab 7 system ID at same `APPROACH_PWM` |

In [ ]:
ble.disconnect()